# MenoPhenotype — Notebook 14: Conclusion

**Course stage covered:** Stage 19 (Conclusion), and doubles as the "Results to be Reported" checklist
(brief Section 13) — every number below is read back from the artifacts the previous 13 notebooks actually
produced, not retyped from memory.


In [1]:
import json
import pandas as pd
from pathlib import Path

ART = Path('artifacts')
qmeta = json.load(open(ART / 'quality_meta.json'))
emeta = json.load(open(ART / 'engineering_meta.json'))
fsets = json.load(open(ART / 'feature_sets.json'))
best_hp = json.load(open(ART / 'best_hyperparameters.json'))
baseline_metrics = json.load(open(ART / 'baseline_model_metrics.json'))
phenotypes = json.load(open(ART / 'phenotype_profiles.json'))
comparison = pd.read_csv(ART / 'model_comparison_table.csv')
final_labels = pd.read_csv(ART / 'final_cluster_labels.csv')
flags = pd.read_csv(ART / 'atypical_profile_flags.csv')
print("Artifacts loaded from", len(list(ART.iterdir())), "files in artifacts/")


Artifacts loaded from 30 files in artifacts/


## A. Dataset summary table (brief Section 13)

In [2]:
dataset_summary = pd.DataFrame([
    ['Source', 'SWAN Baseline (ICPSR 28762) + Cross-Sectional Screener (ICPSR 4368)'],
    ['Citation', 'Sutton-Tyrrell et al. (2019), Inter-university Consortium for Political and Social Research'],
    ['Raw size', '3,302 x 741 (Baseline), 16,142 x 111 (Screener)'],
    ['After quality checks (Notebook 2)', f"{3302 - qmeta['dropped_baseline_rows_n']:,} women retained; "
     f"{qmeta['dropped_baseline_rows_n']} dropped (>30% of 18 symptoms missing); "
     f"{len(qmeta['dropped_baseline_columns'])} columns dropped (structural missingness / zero variance)"],
    ['Train / test split (Notebook 4)', f"{(final_labels.shape[0])} total labelled in Notebook 12 "
     "(2,632 train / 660 test, 80/20 stratified on menopausal stage, seed 42)"],
    ['Screener validation cohort (Notebook 4)', 'independent of Baseline, current-HT users excluded'],
    ['Clustering feature set (Notebook 6)', f"Set {fsets['recommended_set']} -- {len(fsets['sets'][fsets['recommended_set']])} symptom items "
     "(hormones/BMI/age/exercise reserved for post-hoc profiling only)"],
])
dataset_summary.columns = ['Item', 'Value']
dataset_summary


,Item,Value
0,Source,SWAN Baseline (ICPSR 28762) + Cross-Sectional ...
1,Citation,"Sutton-Tyrrell et al. (2019), Inter-university..."
2,Raw size,"3,302 x 741 (Baseline), 16,142 x 111 (Screener)"
3,After quality checks (Notebook 2),"3,290 women retained; 12 dropped (>30% of 18 s..."
4,Train / test split (Notebook 4),"3290 total labelled in Notebook 12 (2,632 trai..."
5,Screener validation cohort (Notebook 4),"independent of Baseline, current-HT users excl..."
6,Clustering feature set (Notebook 6),Set A -- 18 symptom items (hormones/BMI/age/ex...


## B. Model comparison table (final, from Notebook 10)

In [3]:
comparison


,Model,Best Hyperparameters,Validation Result,Test Metric(s),Observations
0,Baseline,"k=2 (sklearn defaults, untuned)",n/a (untuned),"silhouette=0.351, DB=1.847",Reference floor only; separates overall severi...
1,K-Means,k=2,"silhouette 0.360, bootstrap ARI 0.881 (train)","silhouette=0.351, DB=1.847","Fast, simple, best-separated of the four; matc..."
2,Agglomerative,"k=2, linkage=ward","silhouette 0.466 (train, Ward); only balanced ...","silhouette=0.426, DB=1.603",No native out-of-sample predict (nearest-centr...
3,GMM,"k=2, covariance_type=full",bootstrap ARI 0.896 (train); BIC/CV-likelihood...,"silhouette=0.267, DB=2.478",Only model giving soft/probabilistic membershi...
4,DBSCAN (additional),"eps=2.8, min_samples=5","train silhouette 0.348, but the 2nd ""cluster"" ...",test noise=24.0% (silhouette undefined: 0 test...,"Functions as a dense-core-vs-outlier detector,..."


**Reading the test-silhouette column honestly:** K-Means (0.351) and Agglomerative (0.426) actually separate
the test data better than the recommended GMM (0.267). **GMM was still chosen as the primary model**, but for
a reason other than raw separation: it is the only one of the three that gives each woman a *probability* of
belonging to each phenotype rather than a forced hard label — directly matching the blueprint's premise
(Section 2) that menopause symptom profiles overlap rather than falling into clean-edged groups. This is a
deliberate precision-vs-functionality trade-off, not an oversight, and the report should present it as such
rather than silently picking the model with the best single number.


## C. Key findings

In [4]:
findings = pd.DataFrame([
    ['Final structure', '2 phenotypes (not the several domain-specific phenotypes originally hypothesized) -- confirmed independently by K-Means, Agglomerative, and GMM (Notebooks 8-10)'],
    ['Phenotype A', f"{phenotypes['Phenotype A (lower burden)']['n']} women -- lower symptom burden across all 4 domains"],
    ['Phenotype B', f"{phenotypes['Phenotype B (higher, multi-domain burden)']['n']} women -- elevated burden across ALL domains simultaneously, not one dominant domain"],
    ['Boundary drivers (Notebook 12, Random Forest)', 'cold sweats (32% importance), urine leakage, hot flashes, vaginal dryness, night sweats -- a handful of vasomotor/somatic items do most of the work of the split, even though the outcome reads as "overall burden"'],
    ['Robust clinical differences (Bonferroni-corrected, Notebook 12)', 'BMI, waist-to-hip ratio, SHBG, DHEA-S, exercise frequency -- NOT FSH/oestradiol, which looked significant uncorrected but did not survive correction'],
    ['Menopausal stage validation', f"significant association with cluster (p={phenotypes['status_cluster_association_chi2_p']:.2e}), never used in training -- a genuine external validity check"],
    ['Ethics audit', f"race is significantly associated with cluster membership (p={phenotypes['race_cluster_association_chi2_p']:.2e}); BMI/exercise differences are the more defensible explanation than a biological racial difference, but this needs further adjustment analysis before any downstream use"],
    ['Atypical/mixed profiles (Notebook 11)', f"{int((flags['n_flags'] >= 3).sum())} women ({(flags['n_flags'] >= 3).mean():.1%}) flagged by >=3 of 5 independent signals -- elevated across every domain, genuinely heterogeneous rather than a hidden 3rd cluster"],
    ['DBSCAN', 'not usable as a phenotyping model here -- its "2nd cluster" was 5 of 2,632 women; retained only as a dense-core-vs-outlier detector'],
])
findings.columns = ['Topic', 'Finding']
findings


,Topic,Finding
0,Final structure,2 phenotypes (not the several domain-specific ...
1,Phenotype A,2457 women -- lower symptom burden across all ...
2,Phenotype B,833 women -- elevated burden across ALL domain...
3,"Boundary drivers (Notebook 12, Random Forest)","cold sweats (32% importance), urine leakage, h..."
4,Robust clinical differences (Bonferroni-correc...,"BMI, waist-to-hip ratio, SHBG, DHEA-S, exercis..."
5,Menopausal stage validation,significant association with cluster (p=9.63e-...
6,Ethics audit,race is significantly associated with cluster ...
7,Atypical/mixed profiles (Notebook 11),183 women (5.6%) flagged by >=3 of 5 independe...
8,DBSCAN,not usable as a phenotyping model here -- its ...


## D. Methodology recap (notebook-by-notebook)

| # | Notebook | Course stage(s) |
|---|---|---|
| 1 | EDA | Stage 5 |
| 2 | Data Quality Check | Stage 4 |
| 3 | Feature Engineering | Stage 7 |
| 4 | Train-Test Split | Stage 9 |
| 5 | Preprocessing | Stage 6 |
| 6 | Feature Selection | Stage 8 |
| 7 | Baseline Model | Stage 10 |
| 8 | Model Selection & Training | Stages 11-12 |
| 9 | Hyperparameter Tuning | Stage 13 |
| 10 | Model Evaluation & Comparison | Stages 14-15 |
| 11 | Error / Atypical-Profile Analysis | Stage 16 |
| 12 | Interpretation | Stage 17 |
| 13 | Final Demonstration | Stage 18 |
| 14 | Conclusion (this notebook) | Stage 19 |


## E. Limitations

1. **Cross-sectional, single-visit data.** Only SWAN's baseline interview (visit 0) was used; the cohort is
   longitudinal, so the phenotypes describe a snapshot, not how symptom burden changes for a given woman over
   the menopausal transition.
2. **Restricted enrollment population.** SWAN's baseline cohort enrollment criteria excluded women already in
   late perimenopause or post-menopause (Notebook 1) — the 2-phenotype model is only validated for the
   pre-/early-peri range it was trained on. The Screener's broader `STATUS` range was used only for
   cross-walk comparison, not integrated into training.
3. **Coarser structure than hypothesized.** Every clustering method converged on an overall-severity split, not
   the multiple symptom-domain phenotypes (vasomotor-dominant, psychological-dominant, etc.) the project
   blueprint originally proposed. That is a legitimate finding about this feature set and sample, not
   necessarily evidence that domain-specific phenotypes don't exist in menopause more broadly.
4. **No native out-of-sample prediction for 2 of 4 models.** Agglomerative and DBSCAN required manually built
   assignment rules (nearest-centroid, nearest-core-point) rather than a built-in `.predict()` — a reasonable,
   documented workaround (Notebook 10), but an approximation nonetheless.
5. **Unresolved race-cluster association.** Flagged, not explained, in Notebook 12 — needs a proper adjustment
   analysis (controlling for BMI, exercise, and access-to-care proxies) before the phenotypes are used in any
   race-sensitive context.
6. **Weak optimal-k diagnostics in this feature space.** The gap statistic (Notebook 9) never plateaued across
   k=2-8, and BIC/cross-validated GMM likelihood kept improving with more components — neither criterion
   cleanly supported k=2 on its own; the final choice rested on bootstrap stability and cross-algorithm
   agreement instead, which is a reasonable but less standard justification path.
7. **One unverified cross-walk assumption.** The Screener's `DEPRESS` item was paired with Baseline's
   `FEELBLU0` by meaning, not a shared codebook reference (Notebook 3) — flagged but not resolved.
8. **Sample representativeness.** SWAN is a multi-site U.S. cohort recruited in the 1990s; findings may not
   generalise to other populations, eras, or healthcare contexts without re-validation.


## F. Future work

1. **Resolve the race-cluster association** with a formal adjustment analysis (e.g., stratify or regress out
   BMI/exercise/site before re-examining the association) rather than leaving it as an open flag.
2. **Verify the Screener-Baseline item cross-walk** against the ICPSR codebooks, particularly the
   `DEPRESS`/`FEELBLU0` pairing, before using it in any published comparison.
3. **Incorporate SWAN's longitudinal follow-up visits** (beyond baseline) to model how phenotype membership
   *changes* across the menopausal transition for the same women, rather than a single cross-sectional split.
4. **Try methods that don't assume severity dominates**, e.g. feature-weighting schemes, spectral clustering,
   or non-linear embeddings (UMAP) before clustering, to test whether domain-specific sub-structure exists
   beneath the overall-severity signal found here.
5. **Build the self-assessment web app** (Streamlit/Dash, per the blueprint's Section 8/16) using
   `demo_self_assessment.py` (Notebook 13) as the scoring backend.
6. **External replication** on a more recent or more diverse cohort, to check whether the 2-phenotype
   structure and its top drivers (cold sweats, urine leakage, hot flashes, vaginal dryness, night sweats)
   hold outside SWAN.
7. **Clinical outcome validation**, if treatment-response data become available, to test the blueprint's
   original motivating hypothesis directly (Section 3): do the two phenotypes actually respond differently to
   HRT vs. non-hormonal interventions?


## G. Closing summary

The project ran the full pipeline the course brief asks for — problem definition through a working,
standalone demonstration — on a genuinely public, cited health dataset (SWAN, ICPSR 28762/4368), using
unsupervised clustering because no outcome label exists to supervise against. Three independent algorithms
converged on the same reproducible structure (a 2-way, overall-symptom-burden split), which is itself the
project's main empirical result: it says something true about the strongest signal in self-reported SWAN
symptom data, even though it is coarser than the multi-domain phenotype taxonomy originally hypothesized.
Several points in the analysis — the Hopkins/silhouette feature-set comparison (Notebook 6), the
Agglomerative linkage check (Notebook 9), the DBSCAN cluster-balance diagnostic (Notebook 10), and the
Bonferroni-corrected clinical comparison (Notebook 12) — only reached their final, honest conclusion because a
first-pass metric was checked against a second, independent way of looking at the same question rather than
taken at face value. That habit, more than any single number in the model comparison table, is the
methodological takeaway worth carrying into the written report and the viva.
